# 从 Byte-level BPE 到 Hugging Face Tokenizer

> **本章定位**：建立文本、Token ID、特殊 Token 与模型输入之间的稳定协议，并以 Byte-level BPE 贯通算法实现与标准制品。

> **章节边界**：训练语料的大规模清洗与配比见 `A20_data_engineering.ipynb`；WordPiece 在 `E30_nlp_bert.ipynb` 中结合 BERT 协议说明；Encoder、Decoder 与语言模型结构不在本章展开。

> **总览**：内容从上游当前提供的中英语料出发，实现 Byte-level Byte Pair Encoding（字节级字节对编码，BPE）的训练、编码与解码，并迁移到 `tiktoken`、Hugging Face Tokenizers 和 Transformers 标准接口，最终形成可离线重载的 Tokenizer 目录及训练元数据。

```mermaid
flowchart LR
    A["固定版本语料"] --> B["UTF-8 字节"]
    B --> C["BPE Merge Ranks"]
    C --> D["原理编码器"]
    C --> E["tiktoken"]
    E --> F["Fast Tokenizer Backend"]
    F --> G["PreTrainedTokenizerFast"]
    G --> H["BatchEncoding<br/>模型输入"]
```


## 1．学习契约

| 项目 | 内容 |
|---|---|
| 路线 | 共同基础：文本表示 |
| 本章定位 | 解释文本如何转换为稳定 Token ID，并将 Byte-level BPE 原理实现迁移到 Transformers 数据接口。 |
| 先修知识 | 完成 `20_dataset.ipynb`；理解字符串、UTF-8 字节、字典与频次统计。 |
| 预计时间 | 2～3 小时 |
| 运行资源 | CPU 即可；首次运行需要联网下载约 1 MiB 语料。 |
| 输入 | 按数据集 ID 下载并计算文件哈希的 UTF-8 中英文本。 |
| 交付物 | `.tiktoken` ranks、可重载 Hugging Face Tokenizer 目录、训练元数据与编码协议检查结果。 |
| 独立运行 | 直接下载数据仓库当前文件，不依赖上一章的内存或本地产物。 |

### 1.1．学习目标

完成本章后，读者能够解释文本到 Token ID 的转换过程，实现并验证 Byte-level BPE，区分特殊 Token 的角色、字符串、ID 与运行行为，将原理制品迁移到 Transformers 标准接口，并建立可重载、可审计的 Tokenizer 资产契约。


In [ ]:
# 环境：Python 3.11+，regex，tiktoken，Transformers，Tokenizers
# 若环境缺少依赖，请先取消下一行注释并执行：
# %pip install -U datasets huggingface_hub regex tiktoken transformers tokenizers

import base64
import hashlib
import json
import tempfile
from collections import Counter
from pathlib import Path

import datasets
from datasets import load_dataset

import regex
import tiktoken
from tiktoken.load import load_tiktoken_bpe

print(f"tiktoken 已加载：{tiktoken.__file__}")


## 2．直觉与输入输出契约

### 2.1．文本到 Token ID 的转换

Byte-level BPE 先用正则表达式将文本划分为 piece，再将每个 piece 编码为 UTF-8 字节，最后依据训练得到的 merge rank 合并相邻字节块：

Tokenizer 输出的 Token ID 是词表行的整数索引，不是可直接解释为连续大小的数值特征。若将 `[B,L]` 的 ID 显式展开，可得到 `[B,L,V]` 的 One-hot 张量；实际模型使用 Embedding 按 ID 查表，避免物化这个大型稀疏张量。One-hot 定义、与 Embedding/交叉熵的等价性及生产边界见 [10_foundations.ipynb](10_foundations.ipynb)。

```mermaid
flowchart TD
    A["文本 str"] -->|"正则预分词"| B["text pieces"]
    B -->|"UTF-8 encode"| C["byte tokens：0..255"]
    C -->|"按 merge rank 合并"| D["token IDs [L]"]
    D -->|"拼接字节并解码"| E["恢复文本 str"]
```

BPE 训练的每一轮选择当前语料中出现频率最高、且尚未加入词表的相邻 Token 对：

$$
(a^*,b^*)=\arg\max_{(a,b)}\operatorname{count}(a,b)
$$

| 符号或对象 | 含义 | 代码对应 |
|---|---|---|
| $a,b$ | 同一 piece 内相邻的字节块 | `pair`，类型为 `tuple[bytes, bytes]` |
| $\operatorname{count}(a,b)$ | 相邻对在当前语料表示中的出现次数 | `pair_counts` |
| $(a^*,b^*)$ | 当前轮选中的合并对象 | `best_pair` |
| $L$ | 编码后的 Token 数 | `len(token_ids)`，随文本和词表变化 |
| merge rank | Token ID 与编码时的合并优先级 | `mergeable_ranks[token_bytes]` |

初始词表包含全部 256 个字节，因此任意 UTF-8 文本都可以编码，不依赖 `<unk>`。单个字节 Token 可能不位于 Unicode 字符边界；无损解码以完整 Token 序列拼接后的字节流为单位。


### 2.2．训练语料与未见文本边界

语料来自 Hugging Face `google/wmt24pp` 仓库的 `en-zh_CN.jsonl`。下载过程固定文件名并计算 SHA-256；JSONL 约 1.06 MiB、998 行，包含 `source`、`target`、`document_id` 和 `is_bad_source` 等字段，许可证为 Apache-2.0。

数据处理契约如下：

1. 使用 `hf_hub_download()` 获取仓库当前文件并计算完整哈希。
2. 使用 `load_dataset("json")` 读取源文件，剔除 `is_bad_source=True` 的记录并对中英句对精确去重。
3. 按 `document_id` 的 SHA-256 排序划分训练语料与 holdout，使同一文档不会跨越边界。
4. merge ranks 只从训练语料学习；holdout 仅用于未见文本的编码、逐 ID 对照与无损解码检查。

该 holdout 是 Tokenizer 健壮性检查夹具，不承担模型训练评估，也不形成 WMT 翻译基准成绩。缓存完成后，已下载文件可供离线复用。


In [ ]:
from huggingface_hub import hf_hub_download
from tqdm.auto import tqdm

DATASET_ID = "google/wmt24pp"
DATASET_FILENAME = "en-zh_CN.jsonl"
DATASET_LICENSE = "Apache-2.0"
HOLDOUT_FRACTION_DENOMINATOR = 10  # 按文档哈希留出约 10%，且至少保留 1 个文档；修改即形成新数据版本。


def my_locate_project_directory():
    """定位包含课程 Notebook 的项目目录；若当前目录结构不符合预期则抛出 FileNotFoundError。"""
    current_directory = Path.cwd()
    project_directory = current_directory / "llm_from_scratch"
    # Colab 不保证运行目录中存在 ipynb 文件；输出资产只依赖可写目录。
    return project_directory if project_directory.is_dir() else current_directory


PROJECT_DIRECTORY = my_locate_project_directory()


def my_sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    """分块计算文件摘要，并按已读取字节显示校验进度。"""
    resolved = Path(path)
    digest = hashlib.sha256()
    with resolved.open("rb") as handle, tqdm(
        total=resolved.stat().st_size, desc=f"SHA-256 {resolved.name}", unit="B",
        unit_scale=True, unit_divisor=1024, leave=False, dynamic_ncols=True,
    ) as progress:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
            progress.update(len(chunk))
    return digest.hexdigest()


DATASET_PATH = Path(hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename=DATASET_FILENAME,
))
JSONL_SHA256 = my_sha256_file(DATASET_PATH)
RAW_DATASET = load_dataset(
    "json", data_files={"source": str(DATASET_PATH)}, split="source"
)
REQUIRED_COLUMNS = {
    "lp", "document_id", "is_bad_source", "source", "target"
}
missing_columns = REQUIRED_COLUMNS.difference(RAW_DATASET.column_names)
if missing_columns:
    raise ValueError(f"JSONL 缺少必要字段：{sorted(missing_columns)}")
if set(RAW_DATASET["lp"]) != {"en-zh_CN"}:
    raise ValueError("语言对字段不符合 en-zh_CN 数据契约")


def my_is_clean_record(record):
    """检查一条双语记录是否通过来源、许可证和文本质量过滤，返回布尔值。"""
    return (
        record.get("is_bad_source") is False
        and str(record.get("document_id", "")).strip()
        and str(record.get("source", "")).strip()
        and str(record.get("target", "")).strip()
    )


CLEAN_RECORDS = [
    record
    for record in tqdm(
        RAW_DATASET, total=len(RAW_DATASET), desc="清洗 Tokenizer 语料", unit="row", dynamic_ncols=True
    )
    if my_is_clean_record(record)
]
DOCUMENT_IDS = sorted(
    {str(record["document_id"]) for record in CLEAN_RECORDS},
    key=lambda value: hashlib.sha256(value.encode("utf-8")).digest(),
)
if len(DOCUMENT_IDS) < 2:
    raise ValueError("至少需要两个文档才能建立 Tokenizer 训练语料与 holdout 边界")
holdout_document_count = max(
    1, len(DOCUMENT_IDS) // HOLDOUT_FRACTION_DENOMINATOR
)
HOLDOUT_DOCUMENT_IDS = set(DOCUMENT_IDS[:holdout_document_count])

partition_to_pairs = {"training": [], "holdout": []}
partition_to_documents = {"training": set(), "holdout": set()}
seen_pairs = set()
for record in tqdm(
    CLEAN_RECORDS, desc="划分 Tokenizer 语料", unit="row", dynamic_ncols=True
):
    pair = (str(record["target"]).strip(), str(record["source"]).strip())
    if pair in seen_pairs:
        continue
    seen_pairs.add(pair)
    document_id = str(record["document_id"])
    partition = "holdout" if document_id in HOLDOUT_DOCUMENT_IDS else "training"
    partition_to_pairs[partition].append(pair)
    partition_to_documents[partition].add(document_id)

TOKENIZER_TRAINING_PAIRS = partition_to_pairs["training"]
HOLDOUT_PAIRS = partition_to_pairs["holdout"]
CORPUS_SIZES = {
    partition: len(pairs)
    for partition, pairs in partition_to_pairs.items()
}


def my_rows_payload(partition_to_pairs):
    """将各数据分区的句对按稳定顺序序列化为可哈希文本。"""
    return (
        "\n".join(
            json.dumps(
                {"partition": partition, "zh": chinese, "en": english},
                ensure_ascii=False,
                sort_keys=True,
                separators=(",", ":"),
            )
            for partition, pairs in partition_to_pairs.items()
            for chinese, english in pairs
        )
        + "\n"
    ).encode("utf-8")


SELECTED_ROWS_SHA256 = hashlib.sha256(
    my_rows_payload({
        "training": TOKENIZER_TRAINING_PAIRS,
        "holdout": HOLDOUT_PAIRS,
    })
).hexdigest()
TRAINING_ROWS_SHA256 = hashlib.sha256(
    my_rows_payload({"training": TOKENIZER_TRAINING_PAIRS})
).hexdigest()
DATASET_CONTRACT = {
    "id": DATASET_ID,
    "filename": DATASET_FILENAME,
    "license": DATASET_LICENSE,
    "jsonl_sha256": JSONL_SHA256,
    "corpus_sizes": CORPUS_SIZES,
    "selection": {
        "drop_is_bad_source": True,
        "deduplicate_pairs": True,
        "group_key": "document_id",
        "holdout_method": "first_sha256_sorted_documents",
        "holdout_fraction_denominator": HOLDOUT_FRACTION_DENOMINATOR,
        "holdout_document_count": holdout_document_count,
    },
    "implementation": {
        "download": "huggingface_hub.hf_hub_download",
        "loader": "datasets.load_dataset('json')",
        "datasets_version": datasets.__version__,
    },
}

ZH_TO_EN, EN_TO_ZH = "中→英", "英→中"


def my_translation_instruction(direction, source_text):
    """根据翻译方向把源文本包装为统一指令；方向未知时抛出 ValueError。"""
    if direction == ZH_TO_EN:
        return f"翻译成英文：{source_text}"
    if direction == EN_TO_ZH:
        return f"翻译成中文：{source_text}"
    raise ValueError(f"未知翻译方向：{direction}")


def my_expand_bidirectional(base_pairs):
    """把基础中英句对展开成双向翻译样本，并保留方向、输入和目标字段。"""
    examples = []
    for chinese, english in base_pairs:
        examples.append((ZH_TO_EN, chinese, english))
        examples.append((EN_TO_ZH, english, chinese))
    return examples


TOKENIZER_TRAINING_EXAMPLES = my_expand_bidirectional(TOKENIZER_TRAINING_PAIRS)

print(    f"数据集：{DATASET_ID}/{DATASET_FILENAME} | "
    f"datasets={datasets.__version__}"
)
print(
    f"JSONL：{len(RAW_DATASET)} 行 / {DATASET_PATH.stat().st_size / 1024:.1f} KiB "
    f"| sha256={JSONL_SHA256[:12]}… | license={DATASET_LICENSE}"
)
print(
    "Tokenizer 语料边界：",
    {
        "training": len(TOKENIZER_TRAINING_PAIRS),
        "holdout": len(HOLDOUT_PAIRS),
    },
)
print("selected rows sha256：", SELECTED_ROWS_SHA256)


<!-- theory-math-contract:v1 -->
### 2.3．核心机制的语言与数学表达

Tokenizer 把字符串映射为有限词表中的整数序列，再把整数序列恢复为字节或文本。BPE 每轮选择当前语料中频次最高的相邻符号对并合并：

$$
T:\mathcal{S}\rightarrow\{0,\ldots,V-1\}^{L},\qquad
(a^*,b^*)=\arg\max_{(a,b)}\operatorname{count}_{\mathcal C}(a,b)
$$

其中，$\mathcal S$ 是输入字符串集合，$V$ 是词表大小，$L$ 是编码后的 Token 数，$\mathcal C$ 是当前分词后的训练语料。代码中的 `vocab` 对应大小为 $V$ 的 ID 空间，`merges` 对应有序合并规则。Token ID 只是词表行索引，不表示语义距离；模型输入还需通过 Embedding 查表变为 $[B,L,D]$ 的稠密向量。

以上数学表示用于明确变量、形状与约束；实际结论仍需由本章的数值、形状、梯度、性能或失败案例证据验证。

## 3．最小原理实现

### 3.1．BPE 训练、编码与解码

`BPE_PATTERN` 定义正则预分词协议，`BPE_VOCAB_SIZE=400` 只统计普通 Token，四个特殊 Token 在库迁移阶段追加。原理实现由三个对象构成：

| 原理对象 | 职责 | 输入与输出 |
|---|---|---|
| `my_train_byte_pair_ranks` | 从 256 个基础字节逐轮学习合并顺序 | 文本列表 → `dict[bytes, int]` |
| `my_byte_pair_encode_piece` | 按最小 rank 反复合并单个 piece | UTF-8 bytes → Token ID 列表 |
| `MyBPEEncoding` | 对完整文本执行预分词、编码与解码 | `str ↔ list[int]` |

训练阶段按频次降序和字节值升序处理并列候选，使相同语料、正则与实现版本产生稳定 ranks。编码阶段每轮选择 rank 最小的可合并相邻对，直至不存在候选项。


In [ ]:
# 256 个基础字节是 Byte-level BPE 的协议基线，不是训练超参数。
from tqdm.auto import tqdm
# 数字按 1～3 位预分组；语料领域变化时需重新训练并比较序列长度与任务指标。
BPE_PATTERN = r"""[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,3}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""
BPE_VOCAB_SIZE = 400  # 256 个字节加最多 144 次合并；增大会缩短序列但扩大 Embedding 与输出层。
TOKENIZER_TRAINING_TEXTS = [text for pair in TOKENIZER_TRAINING_PAIRS for text in pair]
TOKENIZER_TRAINING_TEXTS += [
    f"{my_translation_instruction(direction, source)}答案：{target}"
    for direction, source, target in TOKENIZER_TRAINING_EXAMPLES
]

def my_train_byte_pair_ranks(texts, vocab_size, pattern):
    """从文本语料训练 byte-level BPE 合并表，返回字节 token 到排名的映射。词表不足或无法继续合并时抛出 ValueError。"""
    if vocab_size < 256:
        raise ValueError("Byte-level BPE 的词表至少要容纳 256 个单字节 token")
    ranks = {bytes([byte_value]): byte_value for byte_value in range(256)}
    pieces = [
        [bytes([byte_value]) for byte_value in piece.encode("utf-8")]
        for text in texts for piece in regex.findall(pattern, text + "\n")
    ]
    merge_budget = max(vocab_size - len(ranks), 0)
    for _ in tqdm(
        range(merge_budget), desc="训练 Byte-level BPE", unit="merge", dynamic_ncols=True
    ):
        # 统计所有 piece 内部的相邻 token 对，跨 piece 不参与合并。
        pair_counts = Counter(pair for piece in pieces for pair in zip(piece[:-1], piece[1:]))
        if not pair_counts:
            break
        ranked_pairs = sorted(pair_counts, key=lambda pair: (-pair_counts[pair], pair))
        best_pair = next(
            (pair for pair in ranked_pairs if pair[0] + pair[1] not in ranks), None
        )
        if best_pair is None:
            break
        # 新 token 的 rank 同时决定其 ID 和后续编码时的合并优先级。
        merged_token = best_pair[0] + best_pair[1]
        ranks[merged_token] = len(ranks)
        merged_pieces = []
        for piece in pieces:
            merged_piece, index = [], 0
            while index < len(piece):
                if index + 1 < len(piece) and (piece[index], piece[index + 1]) == best_pair:
                    merged_piece.append(merged_token)
                    index += 2
                else:
                    merged_piece.append(piece[index])
                    index += 1
            merged_pieces.append(merged_piece)
        pieces = merged_pieces
    return ranks

def my_byte_pair_encode_piece(piece_bytes, mergeable_ranks):
    """依据合并排名对单个字节片段执行 BPE，返回按顺序排列的 token 字节序列。"""
    parts = [bytes([byte_value]) for byte_value in piece_bytes]
    while True:
        # 每轮选择 rank 最小的可合并相邻对，直到没有候选项。
        candidates = [
            (mergeable_ranks.get(parts[index] + parts[index + 1]), index)
            for index in range(len(parts) - 1)
        ]
        candidates = [candidate for candidate in candidates if candidate[0] is not None]
        if not candidates:
            break
        _, merge_index = min(candidates)
        parts[merge_index:merge_index + 2] = [parts[merge_index] + parts[merge_index + 1]]
    return [mergeable_ranks[part] for part in parts]

class MyBPEEncoding:
    """封装最小 byte-level BPE 的普通文本编码与严格 UTF-8 解码契约。"""
    def __init__(self, pattern, mergeable_ranks):
        """保存预分词表达式和合并排名，并建立 token 字节与整数 ID 的双向索引。"""
        self.pattern = pattern
        self.mergeable_ranks = mergeable_ranks
        self.id_to_bytes = {token_id: token_bytes for token_bytes, token_id in mergeable_ranks.items()}

    def encode_ordinary(self, text):
        """对普通文本预分词并执行 BPE，返回整数 token ID 列表。"""
        token_ids = []
        for piece in regex.findall(self.pattern, text):
            token_ids.extend(my_byte_pair_encode_piece(piece.encode("utf-8"), self.mergeable_ranks))
        return token_ids

    def decode_bytes(self, token_ids):
        """把 token ID 序列无损拼接为原始字节串；未知 ID 会触发 KeyError。"""
        return b"".join(self.id_to_bytes[int(token_id)] for token_id in token_ids)

    def decode(self, token_ids):
        """将 token ID 序列严格解码为 UTF-8 文本，无效字节会触发 UnicodeDecodeError。"""
        return self.decode_bytes(token_ids).decode("utf-8", errors="strict")

MERGEABLE_RANKS = my_train_byte_pair_ranks(TOKENIZER_TRAINING_TEXTS, BPE_VOCAB_SIZE, BPE_PATTERN)


## 4．证据验证

### 4.1．原理实现与 `tiktoken` 对照

同一 `BPE_PATTERN` 与 `MERGEABLE_RANKS` 同时构造 `MyBPEEncoding` 和 `tiktoken.Encoding`。`tiktoken` 在本节作为受维护的参考实现，用于核对以下证据：

- 固定中文样例的原理 Token ID 列表与 `tiktoken` Token ID 列表逐项一致。
- 两条路径都能将完整 Token 序列无损恢复为原文。
- 中文字符可能由多个字节 Token 表示，合并结果取决于训练语料和 ranks，而非字符边界。
- 未见词与 Emoji 仍可编码和严格 UTF-8 解码，不需要 UNK 回退。

`tiktoken.Encoding` 的主要参数与原理对象对应如下：

| `tiktoken` 参数 | 原理对象 | 契约 |
|---|---|---|
| `pat_str` | `BPE_PATTERN` | 正则预分词规则 |
| `mergeable_ranks` | `MERGEABLE_RANKS` | 字节块、Token ID 与合并优先级 |
| `special_tokens` | 后续追加的控制符号 | ID 不得与普通词表冲突 |
| `disallowed_special=()` | 普通文本编码策略 | 输入中的特殊 Token 字面量按普通文本处理 |

对照输出需要同时保留原文、正则 pieces、两组 ID、可读 Token 标签和解码结果；只有无损解码而没有逐 ID 对齐，无法证明合并算法等价。


In [ ]:
# 使用同一组 Merge Ranks 构造原理编码器和 tiktoken，以对照两条编码路径。

# 四个特殊 Token 追加在普通词表之后，总 ID 空间为 404；变更时必须同步模型配置与数据协议。
SPECIAL_TOKENS = ["<pad>", "<bos>", "<eos>", "<sep>"]
SPECIAL_TOKEN_TO_ID = {
    token: len(MERGEABLE_RANKS) + index
    for index, token in enumerate(SPECIAL_TOKENS)
}

handwritten_encoding = MyBPEEncoding(BPE_PATTERN, MERGEABLE_RANKS)
library_encoding = tiktoken.Encoding(
    name="transformer_notebook_bpe",
    pat_str=BPE_PATTERN,
    mergeable_ranks=MERGEABLE_RANKS,
    special_tokens=SPECIAL_TOKEN_TO_ID,
)

def my_readable_token(token_bytes):
    # 将可选依赖或平台能力隔离处理，不影响其余验证路径。
    """把 token 字节转换为便于展示的 UTF-8 标签，并为不可打印字节提供转义表示。"""
    try:
        text = token_bytes.decode("utf-8", errors="strict")
        return text.replace(" ", "␠").replace("\n", "↵")
    except UnicodeDecodeError:
        return "".join(f"\\x{byte_value:02x}" for byte_value in token_bytes)

def my_token_labels(encoding, text):
    """编码文本并返回与 token ID 一一对应的可读标签列表。"""
    token_ids = encoding.encode(text, disallowed_special=())
    return [my_readable_token(encoding.decode_single_token_bytes(token_id)) for token_id in token_ids]

sample_text = "我们正在学习中文。"
handwritten_ids = handwritten_encoding.encode_ordinary(sample_text)
library_ids = library_encoding.encode(sample_text, disallowed_special=())
print("原文：", sample_text)
print("正则 pieces：", regex.findall(BPE_PATTERN, sample_text))
print("原理编码 Token IDs：", handwritten_ids)
print("tiktoken IDs：", library_ids)
print("BPE tokens：", my_token_labels(library_encoding, sample_text))
print("解码：", library_encoding.decode(library_encoding.encode(sample_text)))

print("\n中文 token 并不等于单个汉字：")
for text in ["我喜欢数学。", "我们正在学习中文。", "未见词：量子纠缠🧠"]:
    print(f"{text} -> {my_token_labels(library_encoding, text)}")


### 4.2．BPE 合并演化：边界如何逐轮收缩

**学习问题。** BPE 并非直接把字符串切成一组“词”，而是从单字节边界出发，按已学习的 merge rank 逐轮合并。对于上节的真实样例，一个正则 piece 的边界如何变化，最终结果是否与原理编码器一致？

本图从 `sample_text` 的正则 pieces 中选择实际合并轮数最多的一段，使用 `MERGEABLE_RANKS` 重放编码过程。横轴是原 piece 的 UTF-8 字节偏移；每一行是一轮合并后的 token 分区，矩形宽度等于该 token 包含的字节数。图中使用的不是预设示意值，而是当前语料训练得到的 ranks 与真实中间状态。

运行前应明确以下形状与数值不变量：

- 第 0 轮包含 `N` 个单字节 token；每次有效合并后 token 数严格减少 1。
- 任意一轮按顺序拼接全部矩形所代表的 bytes，结果都必须等于原 piece 的 UTF-8 bytes；因此横轴总宽度恒定为 `N`。
- 最后一轮每个 token 都存在于 `MERGEABLE_RANKS`，其 ID 序列必须与 `my_byte_pair_encode_piece()` 的输出逐项一致。
- 单个矩形不一定能独立解码为合法 Unicode；Byte-level BPE 的基本边界是 bytes，而不是字符。


In [ ]:
# 使用当前训练所得 merge ranks 记录真实编码轨迹，不另造合并顺序。
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle

def my_byte_pair_merge_trace(piece_bytes, mergeable_ranks):
    """记录一个字节片段每轮 BPE 合并后的分段状态与所用排名。"""
    parts = [bytes([byte_value]) for byte_value in piece_bytes]
    trace = [{"parts": parts.copy(), "merge_rank": None}]
    while True:
        candidates = [
            (mergeable_ranks.get(parts[index] + parts[index + 1]), index)
            for index in range(len(parts) - 1)
        ]
        candidates = [candidate for candidate in candidates if candidate[0] is not None]
        if not candidates:
            break
        merge_rank, merge_index = min(candidates)
        parts[merge_index:merge_index + 2] = [
            parts[merge_index] + parts[merge_index + 1]
        ]
        trace.append({"parts": parts.copy(), "merge_rank": merge_rank})
    return trace

sample_pieces = regex.findall(BPE_PATTERN, sample_text)
piece_traces = [
    (piece, my_byte_pair_merge_trace(piece.encode("utf-8"), MERGEABLE_RANKS))
    for piece in sample_pieces
]
trace_piece, merge_trace = max(piece_traces, key=lambda item: len(item[1]))
trace_piece_bytes = trace_piece.encode("utf-8")

for step_index, step in enumerate(merge_trace):
    if b"".join(step["parts"]) != trace_piece_bytes:
        raise RuntimeError(f"第 {step_index} 轮破坏了 bytes 可逆性")
    if step_index > 0 and len(step["parts"]) != len(merge_trace[step_index - 1]["parts"]) - 1:
        raise RuntimeError(f"第 {step_index} 轮没有恰好减少一个 token")

final_ids = [MERGEABLE_RANKS[part] for part in merge_trace[-1]["parts"]]
reference_ids = my_byte_pair_encode_piece(trace_piece_bytes, MERGEABLE_RANKS)
if final_ids != reference_ids:
    raise RuntimeError("合并轨迹的最终 IDs 与原理编码器不一致")

figure_height = max(3.6, 0.52 * len(merge_trace) + 1.8)
fig, ax = plt.subplots(figsize=(12, figure_height), constrained_layout=True)
learned_merge_count = max(1, len(MERGEABLE_RANKS) - 256)
for row_index, step in enumerate(merge_trace):
    byte_offset = 0
    for part in step["parts"]:
        token_id = MERGEABLE_RANKS[part]
        if token_id < 256:
            color = "#d9e2ec"
        else:
            color = plt.cm.Blues(0.35 + 0.55 * (token_id - 256) / learned_merge_count)
        rectangle = Rectangle(
            (byte_offset, row_index - 0.38), len(part), 0.76,
            facecolor=color, edgecolor="white", linewidth=1.2,
        )
        ax.add_patch(rectangle)
        label = my_readable_token(part)
        if len(label) > 8:
            label = label[:7] + "…"
        ax.text(
            byte_offset + len(part) / 2, row_index, label,
            ha="center", va="center", fontsize=8, clip_on=True,
        )
        byte_offset += len(part)

ax.set_xlim(0, len(trace_piece_bytes))
ax.set_ylim(-0.7, len(merge_trace) - 0.3)
ax.invert_yaxis()
ax.set_xticks(range(len(trace_piece_bytes) + 1))
ax.set_yticks(range(len(merge_trace)))
ax.set_yticklabels([
    f"轮次 {index}｜{len(step['parts'])} tokens"
    for index, step in enumerate(merge_trace)
])
ax.set_xlabel("原 piece 的 UTF-8 字节偏移")
ax.set_ylabel("合并状态")
ax.set_title(f"真实 BPE 合并演化：{trace_piece!r}")
ax.grid(axis="x", color="#94a3b8", alpha=0.25)
ax.spines[["top", "right", "left"]].set_visible(False)
plt.show()

print(
    "轨迹契约：",
    {
        "piece": trace_piece,
        "utf8_bytes": len(trace_piece_bytes),
        "merge_steps": len(merge_trace) - 1,
        "final_token_count": len(final_ids),
        "final_ids": final_ids,
    },
)


**应观察到的结论。** 所有行覆盖相同的字节区间，而矩形数量随合并轮次单调减少；最终一行的 ID 与原理编码器完全一致。这给出了“词表中的长 token 是由较早 token 递归组合而成”的可观察证据，也说明 BPE 通过改变边界减少序列长度，并未改变原始字节内容。

**不可误读的边界。** 合并先后由当前语料、预分词正则和 rank 共同决定，不代表语义组成关系，也不能据单个样例判断 Tokenizer 的整体质量。图中的 `\x..` 表示某个中间 token 不是独立合法的 UTF-8 字符，这是字节级建模的正常现象；只有完整 token 序列的拼接结果才承担无损解码契约。颜色仅区分基础字节与学习所得 token，不编码注意力、重要性或频率。


## 5．迁移到生产库

### 5.1．固化 `tiktoken` 中间资产

`.tiktoken` 文件使用 `base64(token_bytes) rank` 的逐行格式。它作为转换 Hugging Face fast Tokenizer 的确定性输入，不是后续模型章节的直接依赖。

配套元数据记录正则表达式、特殊 Token、词表规模、训练文本摘要、源数据 ID、许可证与文件 SHA-256。发布过程先写入同目录临时文件，再执行原子替换；重载时由完整哈希验证内容身份。


In [ ]:
# 将 merge ranks 与训练契约原子写入磁盘，并重新加载为本地编码器。

TOKENIZER_NAME = "transformer_notebook_bpe"
RANKS_FILENAME = f"{TOKENIZER_NAME}.tiktoken"
METADATA_FILENAME = f"{TOKENIZER_NAME}.json"

asset_directory = PROJECT_DIRECTORY / "assets" / "tokenizer"
asset_directory.mkdir(parents=True, exist_ok=True)
ranks_path = asset_directory / RANKS_FILENAME
metadata_path = asset_directory / METADATA_FILENAME

ranks_payload = b"".join(
    base64.b64encode(token_bytes) + b" " + str(rank).encode("ascii") + b"\n"
    for token_bytes, rank in sorted(MERGEABLE_RANKS.items(), key=lambda item: item[1])
)
ranks_sha256 = hashlib.sha256(ranks_payload).hexdigest()
training_payload = "\n".join(TOKENIZER_TRAINING_TEXTS).encode("utf-8")
metadata = {
    "schema_version": 1,
    "name": TOKENIZER_NAME,
    "pat_str": BPE_PATTERN,
    "vocab_size": len(MERGEABLE_RANKS) + len(SPECIAL_TOKEN_TO_ID),
    "mergeable_ranks": {
        "file": RANKS_FILENAME,
        "sha256": ranks_sha256,
        "count": len(MERGEABLE_RANKS),
    },
    "special_tokens": SPECIAL_TOKEN_TO_ID,
    "training_contract": {
        "source": "pinned WMT24++ document-grouped tokenizer training corpus and bidirectional instructions only",
        "dataset": {
            **DATASET_CONTRACT,
            "selected_rows_sha256": SELECTED_ROWS_SHA256,
            "training_rows": len(TOKENIZER_TRAINING_PAIRS),
            "training_rows_sha256": TRAINING_ROWS_SHA256,
        },
        "text_count": len(TOKENIZER_TRAINING_TEXTS),
        "sha256": hashlib.sha256(training_payload).hexdigest(),
    },
}

ranks_temporary_path = ranks_path.with_suffix(ranks_path.suffix + ".tmp")
metadata_temporary_path = metadata_path.with_suffix(metadata_path.suffix + ".tmp")
ranks_temporary_path.write_bytes(ranks_payload)
metadata_temporary_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
    encoding="utf-8",
)
ranks_temporary_path.replace(ranks_path)
metadata_temporary_path.replace(metadata_path)

loaded_ranks = load_tiktoken_bpe(str(ranks_path), expected_hash=ranks_sha256)
loaded_metadata = json.loads(metadata_path.read_text(encoding="utf-8"))
reloaded_encoding = tiktoken.Encoding(
    name=loaded_metadata["name"],
    pat_str=loaded_metadata["pat_str"],
    mergeable_ranks=loaded_ranks,
    special_tokens={
        token: int(token_id)
        for token, token_id in loaded_metadata["special_tokens"].items()
    },
)

print("Tokenizer 资产已原子写入并重新加载")
print("  ranks：", ranks_path.relative_to(PROJECT_DIRECTORY))
print("  metadata：", metadata_path.relative_to(PROJECT_DIRECTORY))
print("  sha256：", ranks_sha256)
print("  vocab_size：", metadata["vocab_size"])


### 5.2．特殊 Token 与模型协议

普通 Token 表示原文的字、子词或字节块；特殊 Token（Special Token）承担序列边界、填充、分类或掩码等协议角色。每个特殊 Token 需要同时识别四个维度：

1. **角色**：在当前模型协议中的语义，例如 EOS 表示序列结束。
2. **字面形式**：制品中的字符串，例如 `<eos>`、`</s>` 或 `<|endoftext|>`。
3. **Token ID**：输入模型的整数，通过 `tokenizer.eos_token_id` 等公开属性读取。
4. **行为**：由 Tokenizer 后处理器、模型 `config`、`generation_config` 和训练标签处理共同决定；注册字符串与 ID 不等于自动启用行为。

#### 5.2.1．常见特殊 Token

| 常见写法 | Hugging Face 属性 | 英文全称与作用 | 协议边界 |
|---|---|---|---|
| `<unk>` / `[UNK]` | `unk_token`、`unk_token_id` | Unknown Token（未知 Token）：词表无法表示输入时的回退符号 | WordPiece 等词表可能需要；完整 Byte-level 词表通常不需要 |
| `<pad>` / `[PAD]` | `pad_token`、`pad_token_id` | Padding Token（填充 Token）：将变长序列补成矩形 batch | 对应 `attention_mask=0`；语言模型 label 通常改为 `-100` |
| `<bos>` / `<s>` | `bos_token`、`bos_token_id` | Beginning of Sequence（序列开始 Token） | 注册 BOS 不表示后处理器一定自动插入 |
| `<eos>` / `</s>` | `eos_token`、`eos_token_id` | End of Sequence（序列结束 Token） | 可作为训练终点和生成停止条件；其语义不同于 SEP |
| `<sep>` / `[SEP]` | `sep_token`、`sep_token_id` | Separator Token（分隔 Token） | 表示结构边界，不必然终止生成 |
| `<cls>` / `[CLS]` | `cls_token`、`cls_token_id` | Classification Token（分类 Token） | 常见于 BERT，不是通用 BOS |
| `<mask>` / `[MASK]` | `mask_token`、`mask_token_id` | Mask Token（掩码 Token） | 与控制注意力的 `attention_mask` 不同 |
| `<extra_id_0>`、`<\|user\|>` 等 | `extra_special_tokens`（Transformers 5） | 任务哨兵、聊天角色或工具边界 | Transformers 4 使用 `additional_special_tokens`；语义来自训练数据、`chat_template` 与模型协议 |

#### 5.2.2．不同模型家族的序列协议

| 模型类型 | 常见序列格式 | 常见特殊 Token |
|---|---|---|
| BERT 类 Encoder | `[CLS] A [SEP] B [SEP]` | CLS、SEP、PAD、UNK、MASK；通常没有 BOS/EOS 属性 |
| GPT/Llama 类 Decoder-only | `[BOS] prompt answer [EOS]`，聊天模型另有角色边界 | BOS、EOS；部分原始模型没有 PAD，通常不使用 CLS/MASK |
| T5/BART 类 Encoder–Decoder | source 与 target 分别编码，Decoder 起始 ID 由模型配置决定 | PAD、EOS、可选 BOS/UNK；T5 还包含 `<extra_id_*>` 哨兵 Token |

`decoder_start_token_id` 属于模型配置，用于启动 Decoder，并非所有 Tokenizer 都提供的统一特殊 Token。`token_type_ids` 是与 `input_ids` 同形状的段标识张量；`attention_mask` 和 label 中的 `-100` 也是控制值，不是词表项。

将 `pad_token` 复用为 `eos_token` 只适用于明确允许该协议的模型与场景。即使推理批处理允许复用，训练仍需依靠 `attention_mask` 与 label mask 区分填充位置和真实序列结束。特殊 Token 的实际行为以当前 Tokenizer 属性、后处理模板、模型配置和训练数据协议的联合结果为准。

<!-- diagram:special-token-protocol -->

![架构图：特殊 Token 从文本后处理到模型、生成与训练协议的职责分支](assets/figures/21_tokenizer/special-token-protocol.svg)

[TikZ 源文件](assets/figures/21_tokenizer/special-token-protocol.tex)


### 5.3．接入 Transformers 标准接口

生产项目通常采用两条路径：通过 `AutoTokenizer.from_pretrained()` 加载模型仓库中的现有制品，或将自建的 `tokenizers.Tokenizer` backend 包装为 `PreTrainedTokenizerFast`。本章的自建 BPE 采用第二条路径，转换完成后的调用方只依赖 Hugging Face 标准接口。Transformers 5 中 `PreTrainedTokenizerFast` 是 `TokenizersBackend` 的兼容别名，因此运行时类名显示为 `TokenizersBackend` 属于预期行为。

`TikTokenConverter` 将同一份 ranks 转换为 Hugging Face fast backend，不重新训练词表，因此普通 Token ID 与 `tiktoken` 路径保持一致。wrapper 中注册 `bos_token`、`eos_token` 等属性只建立角色映射；`TemplateProcessing` 才定义 `add_special_tokens=True` 时的实际序列格式。

<!-- diagram:tokenizer-transformers-interface -->

![架构图：BPE ranks 到 Transformers 标准 Tokenizer 制品的分层架构](assets/figures/21_tokenizer/tokenizer-transformers-interface.svg)

[TikZ 源文件](assets/figures/21_tokenizer/tokenizer-transformers-interface.tex)


In [ ]:
# 把底层 BPE backend 包装成 Transformers 可直接使用的 Tokenizer。

from transformers import AutoTokenizer, PreTrainedTokenizerFast
from transformers.integrations.tiktoken import TikTokenConverter
from tokenizers.processors import TemplateProcessing

HF_TOKENIZER_NAME = "transformer_notebook_hf_bpe"
HF_METADATA_FILENAME = "training_metadata.json"
TOKENIZER_MODEL_MAX_LENGTH = 512  # 制品声明的调用上限；不能单独扩大模型的位置容量。
hf_tokenizer_directory = asset_directory / HF_TOKENIZER_NAME
hf_tokenizer_directory.mkdir(parents=True, exist_ok=True)

hf_backend_tokenizer = TikTokenConverter(
    vocab_file=str(ranks_path),
    pattern=BPE_PATTERN,
    add_prefix_space=False,
    extra_special_tokens=SPECIAL_TOKEN_TO_ID,
).converted()
# 特殊 token 注册与自动插入是两层协议；在 backend 中明确定义序列模板。
hf_backend_tokenizer.post_processor = TemplateProcessing(
    single="<bos> $A <eos>",
    pair="<bos> $A <sep> $B:1 <eos>:1",
    special_tokens=[
        ("<bos>", SPECIAL_TOKEN_TO_ID["<bos>"]),
        ("<eos>", SPECIAL_TOKEN_TO_ID["<eos>"]),
        ("<sep>", SPECIAL_TOKEN_TO_ID["<sep>"]),
    ],
)
hf_tokenizer = PreTrainedTokenizerFast(
    tokenizer_object=hf_backend_tokenizer,
    pad_token="<pad>",
    bos_token="<bos>",
    eos_token="<eos>",
    sep_token="<sep>",
    model_max_length=TOKENIZER_MODEL_MAX_LENGTH,
    clean_up_tokenization_spaces=False,
)


# 使用临时目录隔离中间文件，离开作用域后自动清理。
with tempfile.TemporaryDirectory(
    dir=asset_directory, prefix=f".{HF_TOKENIZER_NAME}-"
) as temporary_directory:
    temporary_path = Path(temporary_directory)
    saved_paths = [Path(path) for path in hf_tokenizer.save_pretrained(temporary_path)]
    tokenizer_files = {
        path.name: hashlib.sha256(path.read_bytes()).hexdigest()
        for path in saved_paths
    }
    hf_metadata = {
        "schema_version": 2,
        "format": "huggingface_fast_tokenizer",
        "name": HF_TOKENIZER_NAME,
        "vocab_size": len(hf_tokenizer),
        "special_tokens": SPECIAL_TOKEN_TO_ID,
        "tokenizer_files": tokenizer_files,
        "source_tiktoken": {
            "file": RANKS_FILENAME,
            "sha256": ranks_sha256,
        },
        "training_contract": metadata["training_contract"],
    }
    hf_metadata_path = temporary_path / HF_METADATA_FILENAME
    hf_metadata_path.write_text(
        json.dumps(hf_metadata, ensure_ascii=False, indent=2, sort_keys=True) + "\n",
        encoding="utf-8",
    )
    files_to_publish = saved_paths + [hf_metadata_path]
    for source_path in files_to_publish:
        source_path.replace(hf_tokenizer_directory / source_path.name)

reloaded_hf_tokenizer = AutoTokenizer.from_pretrained(
    hf_tokenizer_directory, local_files_only=True, use_fast=True
)

# 重载后先验证序列协议，避免只检查文件存在。
protocol_ids = reloaded_hf_tokenizer.encode(
    "我们正在学习中文。", add_special_tokens=True
)
if protocol_ids[0] != reloaded_hf_tokenizer.bos_token_id:
    raise RuntimeError("Tokenizer 未按协议插入 BOS")
if protocol_ids[-1] != reloaded_hf_tokenizer.eos_token_id:
    raise RuntimeError("Tokenizer 未按协议插入 EOS")

print("已构建并离线重载 Hugging Face fast tokenizer")
print("  directory：", hf_tokenizer_directory.relative_to(PROJECT_DIRECTORY))
print("  class：", type(reloaded_hf_tokenizer).__name__)
print("  vocab_size：", len(reloaded_hf_tokenizer))
print("  special tokens：", reloaded_hf_tokenizer.special_tokens_map)
print("  protocol tokens：", reloaded_hf_tokenizer.convert_ids_to_tokens(protocol_ids))


### 5.4．Hugging Face Tokenizer 的分层结构

`AutoTokenizer` 是加载工厂，根据本地目录或 Hub 仓库配置选择具体实现。Fast Tokenizer 的公开接口由三层对象协作完成：

| 层级 | 本章对象 | 职责 | 检查入口 |
|---|---|---|---|
| 加载工厂 | `AutoTokenizer` | 根据配置选择实现类 | `from_pretrained()` 参数与仓库文件 |
| Transformers wrapper | `TokenizersBackend`（`PreTrainedTokenizerFast` 兼容别名） | 特殊 Token、padding、truncation、张量类型与保存协议 | `__call__()` 与 `BatchEncoding` |
| fast backend | `tokenizers.Tokenizer` | normalizer → pre-tokenizer → model → post-processor → decoder | `backend_tokenizer` 与 `tokenizer.json` |

稳定的排查顺序是先确认公开输入输出契约，再定位具体 backend 组件和序列化配置。私有方法不构成长期兼容接口。

<!-- diagram:tokenizer-three-layers -->

![架构图：Hugging Face Tokenizer 配置、加载工厂、包装层与 Rust pipeline 分层](assets/figures/21_tokenizer/tokenizer-three-layers.svg)

[TikZ 源文件](assets/figures/21_tokenizer/tokenizer-three-layers.tex)


In [ ]:
import inspect

# 先读 wrapper 的公开契约，再下沉到 fast backend 的组件。
backend_tokenizer = reloaded_hf_tokenizer.backend_tokenizer
print("wrapper class：", type(reloaded_hf_tokenizer).__name__)
print("MRO（Method Resolution Order，方法解析顺序）：", [
    base_class.__name__
    for base_class in type(reloaded_hf_tokenizer).__mro__
])
print("is_fast：", reloaded_hf_tokenizer.is_fast)
print("model_input_names：", reloaded_hf_tokenizer.model_input_names)
print("padding_side：", reloaded_hf_tokenizer.padding_side)
print("model_max_length：", reloaded_hf_tokenizer.model_max_length)
print("__call__ signature：", inspect.signature(
    reloaded_hf_tokenizer.__call__
))

backend_components = {
    "normalizer": backend_tokenizer.normalizer,
    "pre_tokenizer": backend_tokenizer.pre_tokenizer,
    "model": backend_tokenizer.model,
    "post_processor": backend_tokenizer.post_processor,
    "decoder": backend_tokenizer.decoder,
}
for component_name, component in backend_components.items():
    print(f"{component_name:>14}：{type(component).__name__}")

# tokenizer.json 是 fast pipeline 的可执行描述；tokenizer_config.json 保存 wrapper 行为。
tokenizer_json_path = hf_tokenizer_directory / "tokenizer.json"
tokenizer_config_path = hf_tokenizer_directory / "tokenizer_config.json"
tokenizer_document = json.loads(tokenizer_json_path.read_text(encoding="utf-8"))
tokenizer_config_document = json.loads(
    tokenizer_config_path.read_text(encoding="utf-8")
)
print("tokenizer.json 顶层字段：", sorted(tokenizer_document))
print("backend model type：", tokenizer_document["model"]["type"])
print("added token count：", len(tokenizer_document.get("added_tokens", [])))
print("tokenizer class config：", tokenizer_config_document.get("tokenizer_class"))
print("wrapper source：", inspect.getfile(type(reloaded_hf_tokenizer)))


### 5.5．加载 Hub Tokenizer

自建制品用于解释资产形成过程，生产应用更常加载模型仓库已经发布的 Tokenizer。本节按模型 ID 加载 `google-bert/bert-base-multilingual-cased` 当前提供的 Tokenizer 文件。

mBERT 使用 WordPiece，序列协议为 `[CLS] A [SEP] B [SEP]`；本章自建制品使用 Byte-level BPE 与 BOS/EOS 模板。两者的算法和特殊 Token 协议不同，但都通过 `AutoTokenizer` 加载，并返回统一的 `BatchEncoding` 数据结构。


In [ ]:
HUB_TOKENIZER_ID = "google-bert/bert-base-multilingual-cased"

hub_tokenizer = AutoTokenizer.from_pretrained(
    HUB_TOKENIZER_ID,
    use_fast=True,
)
hub_encoding = hub_tokenizer(
    "我们正在学习中文。",
    text_pair="We are learning Chinese.",
    return_offsets_mapping=True,
    return_special_tokens_mask=True,
)

print("Hub ID：", HUB_TOKENIZER_ID)
print("class / fast：", type(hub_tokenizer).__name__, hub_tokenizer.is_fast)
print("backend model：", type(hub_tokenizer.backend_tokenizer.model).__name__)
print("special tokens：", hub_tokenizer.special_tokens_map)
print("tokens：", hub_encoding.tokens())
print("input_ids：", hub_encoding["input_ids"])
print("token_type_ids：", hub_encoding["token_type_ids"])
print("offset_mapping：", hub_encoding["offset_mapping"])


### 5.6．特殊 Token 契约检查

Hugging Face 将常见角色暴露为统一属性。属性值为 `None` 表示当前模型协议没有该角色，并不表示制品损坏。例如，本章 Byte-level BPE 不需要 UNK；mBERT 具有 UNK、CLS 与 MASK，但没有 BOS/EOS。

`special_tokens_map` 给出标准命名角色与字符串的映射。Transformers 5 将额外的非标准角色列表独立为 `extra_special_tokens`；`all_special_ids` 汇总标准角色与额外角色中需要作为控制符号处理的全部 ID。`get_special_tokens_mask()` 或 `return_special_tokens_mask=True` 标记具体序列中由后处理器加入的特殊 Token。对比编码前后的 ID、Token 与 mask，可以验证序列模板是否按预期生效。


In [ ]:
# 特殊 Token ID 由各自制品定义，必须通过公开属性读取，不能跨模型假定固定编号。
SPECIAL_TOKEN_ROLES = ("unk", "pad", "bos", "eos", "sep", "cls", "mask")


def my_print_special_token_contract(tokenizer_name, tokenizer):
    """打印 Tokenizer 各特殊角色对应的 token 文本和 ID，便于审计协议。"""
    print(f"\n{tokenizer_name}")
    for role in SPECIAL_TOKEN_ROLES:
        token = getattr(tokenizer, f"{role}_token")
        token_id = getattr(tokenizer, f"{role}_token_id")
        print(f"  {role:>4}: token={token!r:<12} id={token_id}")
    print("  extra_special_tokens：", tokenizer.extra_special_tokens)
    print("  all_special_tokens：", tokenizer.all_special_tokens)
    print("  all_special_ids：", tokenizer.all_special_ids)


my_print_special_token_contract(
    "自建 Byte-level BPE", reloaded_hf_tokenizer
)
my_print_special_token_contract("mBERT WordPiece", hub_tokenizer)


def my_show_inserted_special_tokens(tokenizer_name, tokenizer, text, text_pair=None):
    """对比启用与禁用自动特殊 token 时的编码结果，并打印新增位置。"""
    without_special_ids = tokenizer(
        text, text_pair=text_pair, add_special_tokens=False
    )["input_ids"]
    encoding_with_specials = tokenizer(
        text,
        text_pair=text_pair,
        add_special_tokens=True,
        return_special_tokens_mask=True,
    )
    print(f"\n{tokenizer_name}")
    print("  原文 token IDs：", without_special_ids)
    print("  加入协议后：", encoding_with_specials["input_ids"])
    print("  tokens：", encoding_with_specials.tokens())
    print("  special mask：", encoding_with_specials["special_tokens_mask"])


my_show_inserted_special_tokens(
    "自建协议：BOS A EOS",
    reloaded_hf_tokenizer,
    "我们正在学习中文。",
)
my_show_inserted_special_tokens(
    "mBERT 句对协议：CLS A SEP B SEP",
    hub_tokenizer,
    "我们正在学习中文。",
    text_pair="We are learning Chinese.",
)

unknown_probe = "🧠"
print("\n未见字符对照：")
print("  Byte-level BPE：", reloaded_hf_tokenizer.tokenize(unknown_probe))
print("  mBERT WordPiece：", hub_tokenizer.tokenize(unknown_probe))


### 5.7．单条文本与 `BatchEncoding`

Hugging Face Tokenizer 提供不同粒度的公开入口：

- `tokenize(text)` 返回 Token 字符串，适合检查切分结果。
- `encode(text)` 返回 Token ID 列表。
- `tokenizer(text, ...)` 返回 `BatchEncoding`，可同时包含 ID、mask、offset 和张量。
- `convert_ids_to_tokens()` 查看 ID 对应的词表项；`decode()` 将完整 ID 序列恢复为文本。

Byte-level BPE 会先把 UTF-8 字节映射到一组可打印 Unicode 符号，因此中文的原始 Token 字符串可能显示为 `ç¿»è¯ĳ...`。这不是编码损坏，也不应逐 Token 当作自然语言阅读；应通过 `offset_mapping` 查看它对应的原文片段，并以完整 ID 序列的 `decode()` round-trip 作为正确性验收。

Fast Tokenizer 的 `offset_mapping` 将 Token 对齐到原文字符区间。特殊 Token 不来自原文，其 offset 通常为 `(0, 0)`。抽取式问答、命名实体识别和高亮显示等任务应基于 offset 契约建立原文映射。


In [ ]:
sample_text = "翻译成英文：我们正在学习中文。"
encoding = reloaded_hf_tokenizer(
    sample_text,
    add_special_tokens=True,
    return_attention_mask=True,
    return_special_tokens_mask=True,
    return_offsets_mapping=True,
)

print("tokenize：", reloaded_hf_tokenizer.tokenize(sample_text))
print("encode：", reloaded_hf_tokenizer.encode(sample_text))
print("BatchEncoding keys：", list(encoding.keys()))
print("input_ids：", encoding["input_ids"])
print("attention_mask：", encoding["attention_mask"])

# 把每个 token 对齐回原文，这是 NER、抽取式 QA 等任务的基础。
for token, token_id, offset, is_special in zip(
    encoding.tokens(),
    encoding["input_ids"],
    encoding["offset_mapping"],
    encoding["special_tokens_mask"],
):
    start, end = offset
    source_fragment = "" if is_special else sample_text[start:end]
    print({
        "token": token,
        "id": token_id,
        "offset": offset,
        "source": source_fragment,
        "special": bool(is_special),
    })

decoded_text = reloaded_hf_tokenizer.decode(
    encoding["input_ids"],
    skip_special_tokens=True,
    clean_up_tokenization_spaces=False,
)
round_trip_ok = decoded_text == sample_text
print("跳过特殊 token 后解码：", decoded_text)
print("round-trip 一致：", round_trip_ok)
if not round_trip_ok:
    raise RuntimeError("Byte-level BPE 的完整 ID 序列未能无损恢复原文。")


### 5.8．批量输入、padding 与 truncation

变长文本需要经过 padding 才能组成形状为 `[batch, sequence]` 的矩形张量。`padding=True` 补齐到当前批次的目标长度，`truncation=True` 允许依据 `max_length` 截断；调用处显式记录这些参数可以避免默认值随制品变化而改变行为。

`attention_mask` 在真实 Token 位置为 1，在 padding 位置为 0。`return_tensors="pt"` 通常与 padding 一起使用。训练数据管线更适合保留变长特征，再由 `DataCollatorWithPadding` 在每个 batch 内动态补齐，从而减少无效 Token 计算。


In [ ]:
from transformers import DataCollatorWithPadding

BATCH_MAX_LENGTH = 64  # 覆盖本章短句；降低会增加截断，提高会增加每批 Token 与内存。
PADDING_MULTIPLE = 8  # 按常见加速器粒度补齐；可能增加 Padding，CPU 不保证收益。

batch_texts = [
    "短句。",
    "这是一条长度明显不同的中英文 mixed sentence。",
    "Emoji 也可以编码：🧠🚀",
]

# 推理时可直接批量编码，得到 [batch, sequence] 张量。
model_inputs = reloaded_hf_tokenizer(
    batch_texts,
    padding=True,
    truncation=True,
    max_length=BATCH_MAX_LENGTH,
    pad_to_multiple_of=PADDING_MULTIPLE,
    return_tensors="pt",
)
for field_name, tensor in model_inputs.items():
    print(field_name, tuple(tensor.shape), tensor.dtype)
print("input_ids:\n", model_inputs["input_ids"])
print("attention_mask:\n", model_inputs["attention_mask"])
print("批量解码：", reloaded_hf_tokenizer.batch_decode(
    model_inputs["input_ids"],
    skip_special_tokens=True,
))

# 训练数据先保留变长列表，collator 只对当前批次动态 padding。
unpadded_features = [
    reloaded_hf_tokenizer(
        text, truncation=True, max_length=64
    )
    for text in batch_texts
]
data_collator = DataCollatorWithPadding(
    tokenizer=reloaded_hf_tokenizer,
    padding=True,
    pad_to_multiple_of=PADDING_MULTIPLE,
    return_tensors="pt",
)
collated_batch = data_collator(unpadded_features)
print("collator input_ids shape：", tuple(collated_batch["input_ids"].shape))


### 5.9．Encoder–Decoder 标签与损失屏蔽

Encoder–Decoder 翻译任务将源文本与目标文本编码为两条序列。`tokenizer(text, text_target=target)` 把目标 Token ID 写入 `labels`，与 Transformers 模型接口保持一致。

`DataCollatorForSeq2Seq` 分别 padding `input_ids` 和 `labels`，并以 `-100` 补齐标签；PyTorch 交叉熵默认忽略 `-100`，因此标签 padding 不参与损失。向 collator 提供具体 Encoder–Decoder 模型后，还可调用模型的 `prepare_decoder_input_ids_from_labels`。

Decoder-only 模型通常把 prompt 与 answer 拼接为一条因果序列，并根据训练目标屏蔽 prompt 标签。两类数据协议不可互换。

<!-- diagram:tokenizer-seq2seq-contract -->
![架构图：Encoder–Decoder 源序列、目标标签与损失屏蔽契约](assets/figures/21_tokenizer/tokenizer-seq2seq-contract.svg)

[TikZ 源文件](assets/figures/21_tokenizer/tokenizer-seq2seq-contract.tex)


In [ ]:
from transformers import DataCollatorForSeq2Seq

LABEL_IGNORE_INDEX = -100  # 交叉熵的损失屏蔽值，不是 Token ID；修改时需同步 ignore_index。

translation_examples = [
    (ZH_TO_EN, "我们正在学习中文。", "We are learning Chinese."),
    (EN_TO_ZH, "Machine learning needs good data.", "机器学习需要高质量数据。"),
]
seq2seq_features = []
for direction, source_text, target_text in translation_examples:
    # text_target 让 wrapper 按目标序列协议产生 labels。
    feature = reloaded_hf_tokenizer(
        my_translation_instruction(direction, source_text),
        text_target=target_text,
        truncation=True,
        max_length=BATCH_MAX_LENGTH,
    )
    seq2seq_features.append(feature)

seq2seq_collator = DataCollatorForSeq2Seq(
    tokenizer=reloaded_hf_tokenizer,
    model=None,
    padding=True,
    label_pad_token_id=LABEL_IGNORE_INDEX,
    return_tensors="pt",
)
training_batch = seq2seq_collator(seq2seq_features)
for field_name, tensor in training_batch.items():
    print(field_name, tuple(tensor.shape), tensor.dtype)

# 解码时先把损失屏蔽值换回 PAD ID，它不是词表 token。
labels_for_decode = training_batch["labels"].masked_fill(
    training_batch["labels"].eq(LABEL_IGNORE_INDEX),
    reloaded_hf_tokenizer.pad_token_id,
)
print("目标文本：", reloaded_hf_tokenizer.batch_decode(
    labels_for_decode, skip_special_tokens=True
))


## 6．生产边界

### 6.1．标准制品目录与重载验收

`save_pretrained()` 保存一组协作文件，而不是单独词表：

- `tokenizer.json`：fast backend 的完整 pipeline，包括 model、pre-tokenizer、post-processor、decoder 与 added tokens。
- `tokenizer_config.json`：Transformers wrapper 的类型、最大长度、特殊 Token 对象和调用行为。
- `training_metadata.json`：本章维护的数据来源、许可证、训练契约与文件哈希；它不是 Transformers 强制文件。

验收需要在独立加载边界比较固定样例的 `input_ids`、特殊 Token 位置、offset、解码结果和文件哈希。`len(tokenizer)` 包含 added tokens，用于确定模型 Embedding 与 LM Head 的行数；模型创建后新增 Token 时，模型需要执行 `resize_token_embeddings(len(tokenizer))` 并重新训练新增行。


In [ ]:
published_files = sorted(
    path for path in hf_tokenizer_directory.iterdir() if path.is_file()
)
print("标准目录：")
for published_path in published_files:
    file_sha256 = hashlib.sha256(published_path.read_bytes()).hexdigest()
    print(" ─", published_path.name, file_sha256[:12] + "…")

verification_texts = [
    "我们正在学习中文。",
    "Unseen text with emoji 🧠",
]
before_save_ids = hf_tokenizer(
    verification_texts, add_special_tokens=True
)["input_ids"]
after_reload_ids = reloaded_hf_tokenizer(
    verification_texts, add_special_tokens=True
)["input_ids"]
if before_save_ids != after_reload_ids:
    raise RuntimeError("保存前后的 token ID 不一致")

print("base vocab_size：", reloaded_hf_tokenizer.vocab_size)
print("len(tokenizer)：", len(reloaded_hf_tokenizer))
print("重载后 ID 一致：", True)


### 6.2．源码定位与维护边界

源码检查应从具体契约问题沿公开调用链定位，避免依赖包内目录结构或私有实现细节：

| 定位目标 | 公开入口 | 实现或配置层 |
|---|---|---|
| 加载类的选择依据 | `AutoTokenizer.from_pretrained()` | `tokenizer_config.json` 的 `tokenizer_class` 与 Auto 映射 |
| padding 与 truncation 的参数解释 | `PreTrainedTokenizerBase.__call__()` | `encode_plus` 与 batch encode 的 fast adapter |
| BPE 切分过程 | `backend_tokenizer` | `tokenizer.json` 中的 pre-tokenizer 与 model |
| BOS/EOS 的插入位置 | `add_special_tokens=True` | backend `post_processor` 的 template |
| ID 到文本的恢复过程 | `decode()` / `batch_decode()` | backend decoder |
| offset 的来源 | `BatchEncoding` | fast backend `Encoding` 保存的 offsets |

代码通过 `inspect` 输出当前环境实际加载的文件与公开签名。版本升级时应以固定回归样例验证行为，而不是依据其他版本的源码推断当前结果。


In [ ]:
from transformers import PreTrainedTokenizerBase

source_targets = [
    ("AutoTokenizer factory", AutoTokenizer),
    ("public tokenizer contract", PreTrainedTokenizerBase),
    ("fast tokenizer adapter", PreTrainedTokenizerFast),
]
for target_name, target_object in source_targets:
    print(f"{target_name}:\n  {inspect.getfile(target_object)}")

public_methods = [
    ("from_pretrained", AutoTokenizer.from_pretrained),
    ("__call__", PreTrainedTokenizerBase.__call__),
    ("decode", PreTrainedTokenizerBase.decode),
    ("batch_decode", PreTrainedTokenizerBase.batch_decode),
]
for method_name, method in public_methods:
    print(f"{method_name}{inspect.signature(method)}")


### 6.3．发布与运行约束

- Hub 资产按 `model_id` 直接加载；本地资产通过完整文件哈希和元数据版本标识。
- Tokenizer 模型 ID、`len(tokenizer)`、特殊 Token ID、`model_max_length`、`padding_side`、`truncation_side` 与模型配置共同构成部署契约。
- padding、truncation、`max_length`、特殊 Token 插入和 `return_tensors` 在调用处显式配置，并纳入固定样例回归。
- 训练管线使用任务对应的 Data Collator，验证 label padding 不参与损失。
- Tokenizer 变更会改变序列长度、Embedding 行语义、缓存键和服务输入，需要与模型权重及数据预处理版本协同发布。
- 自建原理实现与 `tiktoken` 仅保留为算法对照和回归工具；在线编码采用受维护的 fast Tokenizer 实现。
- 生产文本还需要覆盖异常 Unicode、超长输入、特殊 Token 注入、空文本、批量极端长度和多线程并发等测试。
